In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, IntSlider, ToggleButtons, HBox, VBox, HTML, Layout
from IPython.display import display

# ============================================================
# LINEAR-PHASE FIR FILTERS: TYPES I-IV
# ============================================================
#
# This notebook introduces the four classes of real-coefficient
# linear-phase FIR filters.
#
# The theoretical classification depends only on:
#   1. Symmetry or antisymmetry of h[n]
#   2. Odd or even impulse-response length N
#
# Type I   : symmetric, odd N
# Type II  : symmetric, even N
# Type III : antisymmetric, odd N
# Type IV  : antisymmetric, even N
#
# The additional numerical coefficient profile used below is
# NOT part of the theoretical FIR classification. It is only
# used to generate a concrete example for visualization.
# ============================================================

style_html = HTML("""
<style>
.fir-root {width:970px; max-width:970px; font-family:Arial,sans-serif;}
.fir-header {background:linear-gradient(90deg,#075a9c,#1687d9); color:white; padding:10px 15px; border-radius:8px 8px 0 0; font-size:19px; font-weight:bold; letter-spacing:0.2px;}
.fir-intro {background:#f3f8fc; border:1px solid #b9d2e6; border-top:none; padding:8px 12px; border-radius:0 0 8px 8px; font-size:12px; line-height:1.50; margin-bottom:8px;}
.fir-accent {font-weight:bold; color:#075a9c;}
.fir-panel-title {font-size:12.5px; font-weight:bold; margin:0 0 6px 2px; color:#17496f;}
.fir-type-buttons > label, .fir-type-buttons .widget-label {display:none !important; width:0 !important; min-width:0 !important; max-width:0 !important; margin:0 !important; padding:0 !important;}
.fir-type-buttons .widget-toggle-buttons {display:flex !important; flex-direction:row !important; flex-wrap:nowrap !important; width:100% !important; gap:8px !important;}
.fir-type-buttons .widget-toggle-button {flex:1 1 0 !important; min-width:0 !important; height:35px !important; font-weight:bold !important;}
.jupyter-widgets-output-area, .widget-output, .output_area, .output_subarea {overflow-x:visible !important; max-width:none !important;}
</style>
""")

header_html = HTML("""
<div class="fir-root">
<div class="fir-header">Linear-Phase FIR Filters: Types I–IV</div>
<div class="fir-intro">

<span class="fir-accent">Purpose:</span>
this notebook illustrates the four classes of real-coefficient linear-phase FIR filters. The theoretical classification follows directly from the symmetry or antisymmetry of h[n] and from whether the impulse-response length N is odd or even.
<br><br>

<span class="fir-accent">Theoretical properties:</span>
the relations h[n] = h[N-1-n] and h[n] = -h[N-1-n], the Type I–IV classification, the forced zeros at z = +1 and/or z = -1, the reciprocal/conjugate zero structure, and the generalized linear-phase delay (N-1)/2 are the theoretical properties discussed in the accompanying text.
<br><br>

<span class="fir-accent">Numerical demonstration:</span>
the theory does not prescribe particular numerical values for the independent FIR coefficients h[n]. Therefore, in order to display a concrete impulse response, magnitude response, phase response and zero diagram, this notebook generates one representative set of coefficients and then imposes exactly the symmetry or antisymmetry required by the selected FIR type.
<br><br>

<span class="fir-accent">Shape parameter:</span>
Shape is <b>not</b> a theoretical FIR parameter and is not part of the Type I–IV theory. It is an auxiliary parameter introduced only for this interactive demonstration. The independent representative coefficients are generated from the numerical profile <b>exp(-Shape·k/M)[0.55 + 0.15 cos(0.8k)]</b>. Changing Shape therefore changes only the particular numerical example being displayed; it does not change the defining properties of the selected FIR class. The default value Shape = 1.20 and its slider range are arbitrary demonstration choices.
<br><br>

<span class="fir-accent">Additional numerical assumptions:</span>
for Type I the center coefficient of the representative sequence is set to 1.0. The complete impulse response is subsequently normalized so that its largest absolute coefficient is equal to one. These choices are made only for convenient visualization and are not theoretical requirements of FIR filters.
<br><br>

<span class="fir-accent">Frequency response and zeros:</span>
after the representative h[n] has been constructed, H(e<sup>jω</sup>) and all its zeros are calculated directly from those numerical coefficients. Consequently, zeros other than the theoretically forced zeros at z = ±1 belong only to the particular representative example and are not common to every filter of that type.
<br><br>

<span class="fir-accent">Phase display:</span>
the actual phase is displayed using its principal value in the interval [-π, π]. The dashed curve represents the generalized linear-phase backbone. Apparent phase jumps of 2π are therefore a consequence of principal-phase representation and do not indicate loss of the linear-phase property.
<br><br>

<span class="fir-accent">Controls:</span>
N is the impulse-response length and its parity is automatically restricted according to the selected FIR type. Shape changes only the representative numerical coefficient profile. The numerical limits imposed on N and Shape are visualization choices and are not theoretical restrictions.

</div>
</div>
""")

def generate_fir_impulse(fir_type, N, shape):
    if fir_type in ['Type I', 'Type III'] and N % 2 == 0: N += 1
    if fir_type in ['Type II', 'Type IV'] and N % 2 == 1: N += 1
    if fir_type == 'Type I':
        M = (N - 1) // 2
        k = np.arange(M)
        left = np.exp(-shape * k / max(M, 1)) * (0.55 + 0.15 * np.cos(0.8 * k))
        h = np.concatenate((left, np.array([1.0]), left[::-1]))
    elif fir_type == 'Type II':
        M = N // 2
        k = np.arange(M)
        left = np.exp(-shape * k / max(M, 1)) * (0.55 + 0.15 * np.cos(0.8 * k))
        h = np.concatenate((left, left[::-1]))
    elif fir_type == 'Type III':
        M = (N - 1) // 2
        k = np.arange(M)
        left = np.exp(-shape * k / max(M, 1)) * (0.55 + 0.15 * np.cos(0.8 * k))
        h = np.concatenate((left, np.array([0.0]), -left[::-1]))
    else:
        M = N // 2
        k = np.arange(M)
        left = np.exp(-shape * k / max(M, 1)) * (0.55 + 0.15 * np.cos(0.8 * k))
        h = np.concatenate((left, -left[::-1]))
    scale = np.max(np.abs(h))
    if scale > 0: h = h / scale
    return h

def calculate_frequency_response(h, omega):
    n = np.arange(len(h))
    return np.exp(-1j * np.outer(omega, n)) @ h

def calculate_fir_zeros(h):
    threshold = 1e-12
    h_poly = np.array(h, dtype=float)
    nonzero_positions = np.where(np.abs(h_poly) > threshold)[0]
    if len(nonzero_positions) == 0: return np.array([], dtype=complex)
    h_poly = h_poly[nonzero_positions[0]:]
    if len(h_poly) <= 1: return np.array([], dtype=complex)
    return np.roots(h_poly)

def wrap_phase(phi):
    return np.angle(np.exp(1j * phi))

def generalized_phase(fir_type, N, omega):
    delay = (N - 1) / 2.0
    phi = -delay * omega if fir_type in ['Type I', 'Type II'] else np.pi / 2.0 - delay * omega
    return wrap_phase(phi)

def type_characteristics(fir_type):
    if fir_type == 'Type I':
        return {'symmetry':'Symmetric','parity':'Odd','forced_dc':'NO','forced_nyquist':'NO','zero_plus':'NO','zero_minus':'NO','note':'Most general linear-phase FIR type.'}
    if fir_type == 'Type II':
        return {'symmetry':'Symmetric','parity':'Even','forced_dc':'NO','forced_nyquist':'YES','zero_plus':'NO','zero_minus':'YES','note':'Cannot have nonzero gain at omega = pi.'}
    if fir_type == 'Type III':
        return {'symmetry':'Antisymmetric','parity':'Odd','forced_dc':'YES','forced_nyquist':'YES','zero_plus':'YES','zero_minus':'YES','note':'Forced zeros at both frequency endpoints.'}
    return {'symmetry':'Antisymmetric','parity':'Even','forced_dc':'YES','forced_nyquist':'NO','zero_plus':'YES','zero_minus':'NO','note':'DC response is necessarily zero.'}

def plot_fir_type_explorer(fir_type='Type I', N=9, shape=1.20):
    if fir_type in ['Type I', 'Type III'] and N % 2 == 0: N += 1
    if fir_type in ['Type II', 'Type IV'] and N % 2 == 1: N += 1

    h = generate_fir_impulse(fir_type, N, shape)
    n = np.arange(N)
    omega = np.linspace(0.0, np.pi, 1600)
    H = calculate_frequency_response(h, omega)
    magnitude = np.abs(H)
    phase_actual = np.angle(H)
    phase_reference = generalized_phase(fir_type, N, omega)
    zeros = calculate_fir_zeros(h)
    info = type_characteristics(fir_type)
    group_delay = (N - 1) / 2.0

    fig = plt.figure(figsize=(13.4, 8.4))
    grid = fig.add_gridspec(2, 3, width_ratios=[1.42, 1.08, 0.92], height_ratios=[1.00, 1.00], wspace=0.36, hspace=0.88)
    ax_impulse = fig.add_subplot(grid[:, 0])
    ax_mag = fig.add_subplot(grid[0, 1])
    ax_phase = fig.add_subplot(grid[1, 1])
    ax_zero = fig.add_subplot(grid[0, 2])
    ax_info = fig.add_subplot(grid[1, 2])

    markerline, stemlines, baseline = ax_impulse.stem(n, h, basefmt=' ')
    plt.setp(stemlines, linewidth=1.4)
    plt.setp(markerline, markersize=5.5)
    ax_impulse.axhline(0.0, linewidth=0.8)
    ax_impulse.axvline((N - 1) / 2.0, linestyle=':', linewidth=1.2)

    for k in range(N // 2):
        mirror = N - 1 - k
        ax_impulse.plot([k, mirror], [h[k], h[mirror]], linestyle=':', linewidth=0.9, alpha=0.45)

    if N % 2 == 1:
        center = (N - 1) // 2
        ax_impulse.plot(center, h[center], marker='D', markersize=8, label='Center sample')

    ax_impulse.set_xlim(-0.7, N - 0.3)
    ax_impulse.set_ylim(-1.18, 1.18)
    ax_impulse.set_xlabel('Sample index n', fontsize=10.5, labelpad=9)
    ax_impulse.set_ylabel(r'$h[n]$', fontsize=11)
    ax_impulse.set_title(f'{fir_type} Impulse Response\n{info["symmetry"]}, {info["parity"]} Length N = {N}', fontsize=12, pad=12)
    ax_impulse.grid(True, linestyle=':', alpha=0.25)
    if N % 2 == 1: ax_impulse.legend(loc='upper center', bbox_to_anchor=(0.5, -0.09), frameon=False, fontsize=9.8)

    relation = r'$h[n]=h[N-1-n]$' if info['symmetry'] == 'Symmetric' else r'$h[n]=-h[N-1-n]$'
    ax_impulse.text(0.50, 0.97, relation, transform=ax_impulse.transAxes, ha='center', va='top', fontsize=11, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.75))

    ax_mag.plot(omega, magnitude, linewidth=1.8)
    ax_mag.axvline(0.0, linestyle=':', linewidth=1.0)
    ax_mag.axvline(np.pi, linestyle=':', linewidth=1.0)
    ax_mag.set_xlim(0.0, np.pi)
    magnitude_max = max(np.max(magnitude), 1e-6)
    ax_mag.set_ylim(-0.03 * magnitude_max, 1.08 * magnitude_max)
    ax_mag.set_xticks([0.0, np.pi / 2.0, np.pi])
    ax_mag.set_xticklabels(['0', r'$\pi/2$', r'$\pi$'])
    ax_mag.set_xlabel(r'Frequency $\omega$', fontsize=10, labelpad=8)
    ax_mag.set_ylabel(r'$|H(e^{j\omega})|$', fontsize=10)
    ax_mag.set_title('Magnitude Response', fontsize=11.5, pad=10)
    ax_mag.grid(True, linestyle=':', alpha=0.25)

    if info['forced_dc'] == 'YES': ax_mag.plot(0.0, 0.0, marker='X', markersize=8, label=r'Forced zero at $\omega=0$')
    if info['forced_nyquist'] == 'YES': ax_mag.plot(np.pi, 0.0, marker='X', markersize=8, label=r'Forced zero at $\omega=\pi$')

    if info['forced_dc'] == 'YES' and info['forced_nyquist'] == 'YES':
        ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False, fontsize=9.5, columnspacing=1.3, handletextpad=0.5)
    elif info['forced_dc'] == 'YES' or info['forced_nyquist'] == 'YES':
        ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=1, frameon=False, fontsize=9.5)

    valid = magnitude > 1e-5 * magnitude_max
    phase_plot = np.full_like(phase_actual, np.nan)
    phase_plot[valid] = phase_actual[valid]

    ax_phase.plot(omega, phase_plot, linewidth=1.7, label='Actual principal phase')
    ax_phase.plot(omega, phase_reference, '--', linewidth=1.3, label='Generalized linear-phase backbone')
    ax_phase.set_xlim(0.0, np.pi)
    ax_phase.set_ylim(-np.pi, np.pi)
    ax_phase.set_xticks([0.0, np.pi / 2.0, np.pi])
    ax_phase.set_xticklabels(['0', r'$\pi/2$', r'$\pi$'])
    ax_phase.set_yticks([-np.pi, -np.pi / 2.0, 0.0, np.pi / 2.0, np.pi])
    ax_phase.set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])
    ax_phase.set_xlabel(r'Frequency $\omega$', fontsize=10, labelpad=8)
    ax_phase.set_ylabel('Phase [rad]', fontsize=10)
    ax_phase.set_title('Generalized Linear Phase', fontsize=11.5, pad=14)
    ax_phase.grid(True, linestyle=':', alpha=0.25)
    ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.29), ncol=1, frameon=False, fontsize=9.6, labelspacing=0.7)

    theta = np.linspace(0.0, 2.0 * np.pi, 500)
    ax_zero.plot(np.cos(theta), np.sin(theta), '--', linewidth=1.0)
    ax_zero.axhline(0.0, linewidth=0.7)
    ax_zero.axvline(0.0, linewidth=0.7)

    if len(zeros) > 0: ax_zero.plot(np.real(zeros), np.imag(zeros), 'x', markersize=7, markeredgewidth=1.6, label='FIR zeros')

    ax_zero.plot(1.0, 0.0, 'o', markersize=4)
    ax_zero.plot(-1.0, 0.0, 'o', markersize=4)
    ax_zero.text(1.04, 0.05, '+1', fontsize=8.5)
    ax_zero.text(-1.18, 0.05, '-1', fontsize=8.5)

    if info['zero_plus'] == 'YES': ax_zero.plot(1.0, 0.0, marker='X', markersize=9)
    if info['zero_minus'] == 'YES': ax_zero.plot(-1.0, 0.0, marker='X', markersize=9)

    radius = max(1.25, np.max(np.abs(zeros)) * 1.12) if len(zeros) > 0 else 1.25
    radius = min(radius, 4.0)
    ax_zero.set_xlim(-radius, radius)
    ax_zero.set_ylim(-radius, radius)
    ax_zero.set_aspect('equal', adjustable='box')
    ax_zero.set_xlabel(r'$\Re\{z\}$', fontsize=9.5)
    ax_zero.set_ylabel(r'$\Im\{z\}$', fontsize=9.5)
    ax_zero.set_title('Zero Geometry', fontsize=11.5, pad=10)
    ax_zero.grid(True, linestyle=':', alpha=0.25)

    ax_info.axis('off')
    center_text = f'\nCenter sample  : h[{(N - 1) // 2}] = 0' if fir_type == 'Type III' else '\nCenter sample  : independent' if fir_type == 'Type I' else ''
    monitor_text = (
        f'{fir_type.upper()} FIR\n'
        f'────────────────────────\n'
        f'Symmetry       : {info["symmetry"]}\n'
        f'Length N       : {N} ({info["parity"].lower()})\n'
        f'Filter order   : {N - 1}\n'
        f'Group delay    : {group_delay:.1f}\n'
        f'Linear phase   : YES\n'
        f'Forced H(0)=0  : {info["forced_dc"]}\n'
        f'Forced H(pi)=0 : {info["forced_nyquist"]}\n'
        f'Zero at z=+1   : {info["zero_plus"]}\n'
        f'Zero at z=-1   : {info["zero_minus"]}'
        f'{center_text}\n\n'
        f'INTERPRETATION\n'
        f'────────────────────────\n'
        f'{info["note"]}'
    )
    ax_info.text(0.02, 0.97, monitor_text, transform=ax_info.transAxes, ha='left', va='top', fontsize=9.2, family='monospace', linespacing=1.38)

    fig.suptitle(f'FIR Type Explorer — {fir_type}', fontsize=13.5)
    plt.subplots_adjust(left=0.055, right=0.985, top=0.91, bottom=0.14)
    plt.show()
    plt.close(fig)

type_selector = ToggleButtons(options=['Type I', 'Type II', 'Type III', 'Type IV'], value='Type I', description='', button_style='', style={'description_width':'0px'}, layout=Layout(width='100%'))
type_selector.add_class('fir-type-buttons')

N_slider = IntSlider(value=9, min=5, max=21, step=2, description='Length N:', continuous_update=True, style={'description_width':'70px'}, layout=Layout(width='420px'))
shape_slider = FloatSlider(value=1.20, min=0.20, max=3.00, step=0.05, description='Shape:', continuous_update=True, readout_format='.2f', style={'description_width':'55px'}, layout=Layout(width='420px'))

def update_length_parity(change=None):
    current_type = type_selector.value
    current_value = N_slider.value
    if current_type in ['Type I', 'Type III']:
        N_slider.min, N_slider.max, N_slider.step = 5, 21, 2
        if current_value % 2 == 0: current_value += 1
        current_value = max(5, min(21, current_value))
        if current_value % 2 == 0: current_value += 1
    else:
        N_slider.min, N_slider.max, N_slider.step = 4, 20, 2
        if current_value % 2 == 1: current_value += 1
        current_value = max(4, min(20, current_value))
        if current_value % 2 == 1: current_value += 1
    N_slider.value = current_value

type_selector.observe(update_length_parity, names='value')
update_length_parity()

widget_plot = interactive(plot_fir_type_explorer, fir_type=type_selector, N=N_slider, shape=shape_slider)
type_selector.description = ''

plot_output = widget_plot.children[-1]
plot_output.layout = Layout(width='auto', overflow='visible')

type_box = VBox([HTML("<div class='fir-panel-title'>Select FIR class</div>"), type_selector], layout=Layout(width='970px', border='1px solid #b9d2e6', padding='7px 10px', overflow='visible'))

parameter_row = HBox([N_slider, shape_slider], layout=Layout(width='900px', justify_content='space-between', align_items='center'))
parameter_box = VBox([HTML("<div class='fir-panel-title'>Explore the Structure</div>"), parameter_row], layout=Layout(width='970px', border='1px solid #b9d2e6', padding='7px 10px', overflow='visible'))

main_layout = VBox([header_html, type_box, parameter_box, plot_output], layout=Layout(width='970px', overflow='visible', align_items='flex-start'))

display(style_html)
display(main_layout)